In [1]:
from langchain.document_loaders import PyPDFLoader, DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

In [7]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("../data/Medical_book.pdf")
extracted_data = loader.load()

In [8]:
len(extracted_data)

637

In [9]:
from typing import List
from langchain.schema import Document

def filter_to_minimal_docs(docs: List[Document]) -> List[Document]:
    """
    Given a list of document objects, return a new list of document objects containing only 'source' in metdata and the original page_content
    """

    minimal_docs: List[Document] = []

    for doc in docs:
        src = doc.metadata.get("source")

        minimal_docs.append(
            Document(
                page_content=doc.page_content,
                metadata={"source": src}
            )
        )

    return minimal_docs

In [10]:
minimal_docs = filter_to_minimal_docs(extracted_data)

In [11]:
def text_split(minimal_docs):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=20,
    )
    texts_chunk = text_splitter.split_documents(minimal_docs)
    return texts_chunk

In [12]:
text_chunk = text_split(minimal_docs)
print(f"Number of Chunks: {len(text_chunk)}")

Number of Chunks: 5859


In [13]:
from langchain_community.embeddings import HuggingFaceEmbeddings

def download_embedding():
    """
    Download and return the HuggingFace embeddings model
    """

    model_name = "sentence-transformers/all-MiniLM-L6-v2"
    embeddings = HuggingFaceEmbeddings(
        model_name = model_name
    )

    return embeddings

embedding = download_embedding()

C:\Users\DELL\AppData\Local\Temp\ipykernel_10644\2552162825.py:9: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


In [14]:
embedding

HuggingFaceEmbeddings(client=SentenceTransformer(
  (0): Transformer({'max_seq_length': 256, 'do_lower_case': False}) with Transformer model: BertModel 
  (1): Pooling({'word_embedding_dimension': 384, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
), model_name='sentence-transformers/all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, multi_process=False, show_progress=False)

In [15]:
vector = embedding.embed_query("Hello World")

In [16]:
print(f"Length of Vector: {len(vector)}")

Length of Vector: 384


In [18]:
from dotenv import load_dotenv
import os
load_dotenv()

True

In [33]:
MEDICALBOT_PINECONE_API_KEY = os.getenv("MEDICALBOT_PINECONE_API_KEY")
GOOGLE_GEN_AI = os.getenv("GOOGLE_GEN_AI")

os.environ["MEDICALBOT_PINECONE_API_KEY"] = MEDICALBOT_PINECONE_API_KEY
os.environ["GOOGLE_GEN_AI"] = GOOGLE_GEN_AI

In [20]:
from pinecone import Pinecone 
pinecone_api_key = MEDICALBOT_PINECONE_API_KEY

pc = Pinecone(api_key=pinecone_api_key)

In [21]:
pc

In [22]:
from pinecone import ServerlessSpec 

index_name = "medical-chatbot"

if not pc.has_index(index_name):
    pc.create_index(
        name = index_name,
        dimension=384,  # Dimension of the embeddings
        metric= "cosine",  # Cosine similarity
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )


index = pc.Index(index_name)

In [24]:
from langchain_pinecone import PineconeVectorStore

docsearch = PineconeVectorStore.from_documents(
    documents=text_chunk,
    embedding=embedding,
    index_name=index_name
)

In [25]:
from langchain_pinecone import PineconeVectorStore
# Embed each chunk and upsert the embeddings into your Pinecone index.
docsearch = PineconeVectorStore.from_existing_index(
    index_name=index_name,
    embedding=embedding
)

In [26]:
dummyDoc = Document(
    page_content="Hi this is Ninad More, a software engineer at Barclays",
    metadata={"source": "LinkedIn"}
)

In [27]:
docsearch.add_documents(documents=[dummyDoc])

['aae473a6-66f6-4a36-9288-8d9154d56691']

In [28]:
retriever = docsearch.as_retriever(search_type="similarity", search_kwargs={"k":3})

In [29]:
retrieved_docs = retriever.invoke("What is Acne?")

In [30]:
retrieved_docs

[Document(id='3259cb99-3f7d-427f-903b-f6ac938de8a4', metadata={'source': '../data/Medical_book.pdf'}, page_content='GALE ENCYCLOPEDIA OF MEDICINE 226\nAcne\nGEM - 0001 to 0432 - A  10/22/03 1:41 PM  Page 26'),
 Document(id='9cc02f40-aecb-4762-8a6d-3fbfdc0a0529', metadata={'source': 'data\\Medical_book.pdf'}, page_content='GALE ENCYCLOPEDIA OF MEDICINE 226\nAcne\nGEM - 0001 to 0432 - A  10/22/03 1:41 PM  Page 26'),
 Document(id='8bbaa41f-3a40-40fb-9dc8-ef557fc62979', metadata={'source': 'data\\Medical_book.pdf'}, page_content='GALE ENCYCLOPEDIA OF MEDICINE 226\nAcne\nGEM - 0001 to 0432 - A  10/22/03 1:41 PM  Page 26')]

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

chatModel = ChatGoogleGenerativeAI(
    model="gemini-3.8-flash",
    google_api_key=GOOGLE_GEN_AI,
    temperature=0
)

In [57]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

In [58]:
system_prompt = (
    "You are an Medical assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to asnwer "
    "the question. If you don't know the answer, say that you "
    "don't know. Use the three sentences maximum and keep the "
    "answer consice."
    "\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)

In [59]:
question_answer_chain = create_stuff_documents_chain(chatModel, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

In [60]:
response = rag_chain.invoke({"input": "what is Acromegaly and gigantism?"})
print(response["answer"])

Based on the provided context, acromegaly is a disorder where the abnormal release of a chemical from the pituitary gland in the brain causes increased growth in bone and soft tissue, as well as other disturbances throughout the body. The provided text does not contain a specific definition for gigantism.
